# Phase 1 — Raw Data Cleaning

**Goal:** Load the raw Excel dataset, fix structural errors (misaligned columns,
wrong data in wrong columns, feature-phone rows), and export a clean CSV for Phase 2.

**Input:** `data/Smart-phone.xlsx`  
**Output:** `data/Clean1_Smart_phone.csv`

## 1. Import Libraries & Load Data

In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
df  = pd.read_excel("data\Smart-phone.xlsx")

## 2. Extract Brand Column

Derive `Company` from the first word of the `model` column, then normalise the `OPPO` → `Oppo` inconsistency.

In [ ]:

df["Company"] = df["model"].apply(lambda x : x.split(" ")[0])

In [ ]:
df["Company"].replace({"OPPO":"Oppo"},inplace=True)
df["Company"].value_counts()

## 3. Filter & Remove Outlier Rows

Drop rows with price < ₹3,400 (non-smartphone / data-entry errors) and known bad rows with column misalignment.

In [ ]:
df1  = df.copy()

In [ ]:
df[df["price"]==df["price"].max()]

In [ ]:
df1  = df1[df1["price"]>=3400]

In [ ]:
df1["price"].describe()

### 3.1 Drop Rows with Corrupt / Misaligned Data

Rows 786, 974, 979, 985 are feature phones (not Android smartphones) with severely misaligned columns. Row 529 has a corrupt processor entry.

In [ ]:
df1.drop([786,974,979,985,529],inplace=True)

In [ ]:
print(df1[df1["Company"].isin(["Nokia"])& (df1["price"] < 4000)])
df1.drop([988],inplace=True)

## 4. Fix Misaligned Columns

Some rows have scraping artefacts causing data to shift right by one column. Detected by checking if `battery` contains `inches` (display data in wrong column).

#### Extract the phones which have inches in battery by wrong

In [ ]:

temp_df = df1[df1["battery"].apply(lambda x : x.split(" ")[1].strip()).str.contains("inch")]

In [ ]:
x  = temp_df.iloc[:,7:-1].shift(1,axis=1).values

In [ ]:
x_index = temp_df.index
df1.loc[temp_df.index,temp_df.columns[7:-1]] = x 
df1.isnull().sum()

### 4.1 Remove Remaining Feature Phones

Nokia 2720 Flip, 2780 Flip, 2660 Flip, 225 4G are feature phones incompatible with the smartphone dataset.

In [ ]:
df1[df1["Company"].str.contains("Nokia")]["model"].to_dict()

## 5. Fix Camera / Card Column Misalignment

26 rows have display data (`inches …`) in the `camera` column and camera data shifted into `card`. Correct by shifting left.

#### Removing features phones

In [ ]:
df1.drop([963, 978, 982, 989],inplace = True)

#### Phone which have issue in camera wrong data -> Display

## 6. Fix OS / Card Column Misalignment

Rows where `card` contains OS data (Android/iOS version strings) need their `os` column restored via a right-shift.

In [ ]:
temp_df = df1[df1["camera"].str.contains("Display")]
temp_df.shape

In [ ]:
df1.loc[temp_df.index,'camera'] = temp_df["card"].values

*All `card` values are now valid memory-card strings.*

In [ ]:
temp_df2 = df1[df1["card"].str.contains("MP")]
df1.loc[temp_df2.index,'card'] = 'Memory Card Not Supported'
df1["card"].value_counts()

### 6.1 Fix Residual OS Misalignment

14 rows still have card-related strings in the `os` column due to the previous shift. Apply a corrective left-shift.

In [ ]:
temp_df3 = df1[~df1["card"].str.contains("Card")]
x  = temp_df3.iloc[:,10:-1].shift(1,axis=1)
df1.loc[temp_df3.index,'os'] = x
df1.loc[temp_df3.index,'card'] = "Memory Card Not Supported"

In [ ]:
df1['os'].value_counts()

#### All card value is perfect

## 7. Export Clean CSV

> **Note on `rating`:** `rating` is a post-sale user review score. It is retained in this CSV for EDA purposes (Phase 3) but will be **dropped in Phase 2** before model-ready feature engineering to prevent data leakage.

In [ ]:
df1['card'].value_counts()

### Quick Sanity Check

#### Wrong card value in os column due to shifting